# Text Mining Assignment 1

### Loading Dataset

In [ ]:
from sklearn.datasets import fetch_20newsgroups

categories = [
    "alt.atheism",
    "comp.graphics", 
    "comp.os.ms-windows.misc", 
    "comp.sys.ibm.pc.hardware", 
    "comp.sys.mac.hardware", 
    "comp.windows.x", 
    "misc.forsale", 
    "rec.autos", 
    "rec.motorcycles", 
    "rec.sport.baseball", 
    "rec.sport.hockey", 
    "sci.crypt", 
    "sci.electronics",
    "sci.med", 
    "sci.space", 
    "soc.religion.christian", 
    "talk.politics.guns", 
    "talk.politics.mideast", 
    "talk.politics.misc",
    "talk.religion.misc",
]


def size_mb(docs):
    return sum(len(s.encode("utf-8")) for s in docs) / 1e6

'\ndef load_dataset(verbose=False, remove=()):\n    Load and vectorize the 20 newsgroups dataset.\n\n\n\n\n    # Extracting features from the training data using a sparse vectorizer\n    t0 = time()\n    vectorizer = TfidfVectorizer(\n        sublinear_tf=True, max_df=0.5, min_df=5, stop_words="english"\n    )\n    X_train = vectorizer.fit_transform(data_train.data)\n    duration_train = time() - t0\n\n    # Extracting features from the test data using the same vectorizer\n    t0 = time()\n    X_test = vectorizer.transform(data_test.data)\n    duration_test = time() - t0\n\n    feature_names = vectorizer.get_feature_names_out()\n\n\n\n        print(\n            f"vectorize training done in {duration_train:.3f}s "\n            f"at {data_train_size_mb / duration_train:.3f}MB/s"\n        )\n        print(f"n_samples: {X_train.shape[0]}, n_features: {X_train.shape[1]}")\n        print(\n            f"vectorize testing done in {duration_test:.3f}s "\n            f"at {data_test_size_mb / 

First we load the dataset utilising the heuristics provided by sklearn to remove message metadata:

In [4]:
from time import time

data_train = fetch_20newsgroups(
    subset="train",
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=("headers", "footers", "quotes"),
)

data_test = fetch_20newsgroups(
    subset="test",
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=("headers", "footers", "quotes"),
)

target_names = data_train.target_names

data_train_size_mb = size_mb(data_train.data)
data_test_size_mb = size_mb(data_test.data)

print(
    f"{len(data_train.data)} documents - "
    f"{data_train_size_mb:.2f}MB (training set)"
)
print(f"{len(data_test.data)} documents - {data_test_size_mb:.2f}MB (test set)")
print(f"{len(target_names)} categories")

11314 documents - 13.78MB (training set)
7532 documents - 8.26MB (test set)
20 categories


Since the target doesn't need to get vectorizer, we can extract them now.

In [5]:
y_train, y_test = data_train.target, data_test.target

### Vectorizer Function

Next we define a function to vectorize the train and test data based on three types of features: counts, tf, and tf-idf. 

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

def vectorize_dataset(data_train, data_test, feature="count", **vectorizer_params):
    """ Vectorise dataset based on certain feature.
    
    feature: "count", "tf", or "tfidf". Decides which sklearn vectorizer is used. 
    vectorizer_params: optional params passed to the sklearn vectorizer 
    """
    if feature == "count":
        vectorizer = CountVectorizer(**vectorizer_params)
    elif feature == "tf":
        vectorizer = TfidfVectorizer(use_idf=False, **vectorizer_params)
    elif feature == "tfidf":
        vectorizer = TfidfVectorizer(use_idf=True, **vectorizer_params)
    else:
        raise ValueError(f"feature must be 'count', 'tf' or 'tfidf', got {feature}")
        
    
    t0 = time()
    X_train = vectorizer.fit_transform(data_train.data)
    duration_train = time() - t0
    
    t0 = time()
    X_test = vectorizer.transform(data_test.data)
    duration_test = time() - t0
    
    feature_names = vectorizer.get_feature_names_out()
    
    print(
        f"vectorize training done in {duration_train:.3f}s "
        f"at {data_train_size_mb / duration_train:.3f}MB/s"
    )
    print(f"n_samples: {X_train.shape[0]}, n_features: {X_train.shape[1]}")
    print(
        f"vectorize testing done in {duration_test:.3f}s "
        f"at {data_test_size_mb / duration_test:.3f}MB/s"
    )
    print(f"n_samples: {X_test.shape[0]}, n_features: {X_test.shape[1]}")
    
    return X_train, X_test, feature_names, vectorizer

### Comparing Classifiers 

In this section we will compare three classification models

vectorize training done in 0.872s at 15.799MB/s
n_samples: 11314, n_features: 101631
vectorize testing done in 0.571s at 14.466MB/s
n_samples: 7532, n_features: 71018


### Comparing Features (count, tf, tfidf) Across Classifiers

In this section we will compare three features i.e. methods of vectorising the train/test datasets, by using the three classification models from the previous section.

### TfidfVectorizer Params Experimentation

In this section we will narrow in on the TfidfVectorizer, experimenting with its different parameters to see which one performs the best.